# 🧠 EMS741 Exam Prep: Ollama & Study Environment
This notebook installs Ollama natively into your Conda/Mamba environment (with GPU acceleration), starts the server, and provides a structured workspace for your bookclub-style exam focusing on Lagergren (2020) and Li (2020).

In [1]:
# 1. Install dependencies natively via Conda/Pip (Forcing CUDA/GPU variant)
!mamba install -y -c conda-forge ollama "ollama=*=*cuda*"
!pip install ollama -q

import subprocess
import time
import ollama

# 2. Start the background server
print("Starting GPU-enabled Ollama server...")
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 3. Pull the model
MODEL_NAME = 'llama3'
print(f"Pulling {MODEL_NAME}...")
!ollama pull {MODEL_NAME}
print("✅ Setup Complete!")


Looking for: ['ollama', 'ollama=[build=*cuda*]']

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache

Pinned packages:
  - python 3.11.*


Transaction

  Prefix: /home/jovyan/env/mamba-gpu

  All requested packages already installed

Starting GPU-enabled Ollama server...
Pulling llama3...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 6a0746a1ec1a: 100% ▕██████████████████▏ 4.7 GB                         
pulling 4fa551d4f938: 100% ▕██████████████████▏  12 KB                         
pulling 8ab4849b038c: 100% ▕██████████████████▏  254 B                         
pulling 577073ffcc6c: 100% ▕██████████████████▏  110 B                         
pulling 3f8eb4da87fa: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 
✅ Setup Complete!


In [2]:
def query_exam_tutor(prompt, system_context="You are an expert AI tutor helping a robotics MSc student prepare for their EMS741 exam."):
    """Streams the answer from Ollama using the official python client."""
    print(f"\n📝 Question: {prompt}\n")
    print("🤖 Answer:")
    
    response = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'system', 'content': system_context},
        {'role': 'user', 'content': prompt}
    ], stream=True)
    
    for chunk in response:
        print(chunk['message']['content'], end='', flush=True)
    print("\n\n" + "━"*60)


## 🧪 Paper 1: Lagergren 2020 (BINNs)
**Core concepts:** Physics-informed loss, learning unknown PDE terms (diffusivity/growth), scratch-assays, time-delay term, constraints/priors.

In [3]:
ctx_lagergren = """
Context: Lagergren 2020 'Biologically-informed neural networks'.
BINNs learn unknown reaction-diffusion PDE terms directly from sparse scratch-assay data. 
They use an MLP surrogate for density u(x,t), and MLPs for parameters D(u) and G(u). 
They added a time-delay term T(t) to handle initial confluency discrepancy.
"""

query_exam_tutor(
    "Describe the main building blocks of the BINN model (inputs, outputs, architecture, loss function).", 
    system_context=ctx_lagergren
)


📝 Question: Describe the main building blocks of the BINN model (inputs, outputs, architecture, loss function).

🤖 Answer:
The Biologically-Informed Neural Network (BINN) model, proposed by Lagergren in 2020, is designed to learn unknown reaction-diffusion PDE terms directly from sparse scratch-assay data. The main building blocks of the BINN model are:

**Inputs:**

* Scratch-assay data (a sparse matrix containing information about cell behaviors)
* Time delay term T(t) (to account for initial confluency discrepancy)

**Outputs:**

* Density u(x,t) (an MLP surrogate to predict the density distribution at each point in space and time)
* Parameters D(u) and G(u) (MLPs to predict the diffusion coefficient and reaction rate, respectively, which are functions of the density u)

**Architecture:**

1. **MLP Surrogate for Density**: A multi-layer perceptron (MLP) is used as a surrogate model to predict the density distribution u(x,t). The input to this MLP is the scratch-assay data.
2. **Tim

## 🧪 Paper 2: Li 2020 (CNN Surrogate)
**Core concepts:** Encoder-decoder CNN, 2D reaction-diffusion, purely data-driven (trained on 1M FEM samples), no physics in loss, ~300x speedup.

In [4]:
ctx_li = """
Context: Li 2020 'CNN surrogate for reaction-diffusion'.
A CNN encoder-decoder predicts 2D concentration fields bypassing FEM.
Inputs: 4-channel tensor (geometry/BC, D, K, time) as 21x21 matrices.
Data-driven (MSE loss on synthetic FEM data), no physical constraints in loss.
"""

query_exam_tutor(
    "What are the inputs, outputs, and architecture of this CNN model? How does it handle time-dependence?", 
    system_context=ctx_li
)


📝 Question: What are the inputs, outputs, and architecture of this CNN model? How does it handle time-dependence?

🤖 Answer:
Based on the context, here's a breakdown of the input, output, and architecture of the CNN model:

**Inputs:**

* A 4-channel tensor with shape (21x21) representing:
	1. Geometry/BC (boundary conditions)
	2. Diffusion coefficient (D)
	3. Reaction rate constant (K)
	4. Time
* Each channel is a separate 2D matrix, resulting in a 4-channel tensor.

**Output:**

* The predicted 2D concentration field as a single 2D matrix with shape (21x21)

**Architecture:**

The model architecture appears to be an encoder-decoder design:

1. **Encoder:** A convolutional neural network (CNN) that processes the input data, likely using convolutional and pooling layers to extract features.
2. **Decoder:** Another CNN or a series of fully connected layers that generate the predicted concentration field.

**Time-Dependence Handling:**

The model does not explicitly handle time-dependen

## ⚖️ Cross-Paper Comparison
**Core concepts:** Physics-informed (BINNs) vs Physics-based training data (Li CNN).

In [5]:
ctx_compare = """
You are comparing BINNs (Lagergren, 2020) and CNN surrogates (Li, 2020).
Lagergren: Physics in the loss function, continuous MLP, sparse real data.
Li: Physics only in the training data generation, discrete 2D CNN, massive synthetic data.
"""

query_exam_tutor(
    "Compare how these two papers incorporate physical laws. Which one enforces physical constraints during training, and how?", 
    system_context=ctx_compare
)


📝 Question: Compare how these two papers incorporate physical laws. Which one enforces physical constraints during training, and how?

🤖 Answer:
In the paper by Lagergren (2020), the authors use a continuous multi-layer perceptron (MLP) to model the underlying physics of the problem. To enforce physical constraints during training, they incorporate the physical laws into the loss function. Specifically, they add penalty terms that ensure the output of the network satisfies certain physical constraints, such as conservation of energy or momentum. This approach is known as a "physics-informed" neural network (PINN).

In contrast, Li et al. (2020) do not enforce physical constraints during training in the same way. Instead, they generate synthetic data that satisfies the underlying physical laws and use this data to train a discrete 2D convolutional neural network (CNN). The physical laws are only used to generate the synthetic data, rather than being incorporated directly into the loss 